In [25]:
import nflreadpy as nfl
import pandas as pd
# Load weekly player stats for 2019-2025
# 2018 used only as historical context for 2019 features
player_stats = nfl.load_player_stats(range(2018,2026))

# Convert from Polars to pandas
df = player_stats.to_pandas()

# Only use regular-season games for fantasy modeling
df = df[df['season_type'] == 'REG'].copy()

players = nfl.load_players().to_pandas()

player_metadata = players[
    ['gsis_id', 'rookie_season']
].copy()

player_metadata = player_metadata.rename(
    columns={'gsis_id': 'player_id'}
)

df = df.merge(
    player_metadata,
    on='player_id',
    how='left'
)

player_seasons = (
    df[
        ['player_id', 'season']
    ]
    .dropna(subset=['player_id'])
    .drop_duplicates()
    .sort_values(['player_id', 'season'])
)

player_seasons['previous_active_season'] = (
    player_seasons
    .groupby('player_id')['season']
    .shift(1)
)

player_seasons['season_gap'] = (
    player_seasons['season']
    - player_seasons['previous_active_season']
)

df = df.merge(
    player_seasons,
    on=['player_id', 'season'],
    how='left'
)

df['is_rookie'] = (
    df['season'] == df['rookie_season']
).astype(int)

df['is_returning_veteran'] = (
    (df['season'] > df['rookie_season']) &
    (
        df['previous_active_season'].isna() |
        (df['season_gap'] > 1)
    )
).astype(int)

player_season_targets = (
    df.groupby(
        ['player_id', 'season'],
        as_index=False
    )['targets']
    .mean()
    .rename(
        columns={'targets': 'season_targets_avg'}
    )
)

previous_season_targets = (
    player_season_targets.rename(
        columns={
            'season': 'previous_active_season',
            'season_targets_avg': 'returning_targets_prior'
        }
    )
)

df = df.merge(
    previous_season_targets,
    on=[
        'player_id',
        'previous_active_season'
    ],
    how='left'
)

rookie_prior_rows = []

for season in range(2019, 2026):

    historical_rookies = df[
        (df['season'] < season) &
        (df['is_rookie'] == 1)
    ]

    position_priors = (
        historical_rookies
        .groupby('position')['targets']
        .mean()
    )

    for position, prior in position_priors.items():
        rookie_prior_rows.append({
            'season': season,
            'position': position,
            'rookie_targets_prior': prior
        })

rookie_targets_priors = pd.DataFrame(
    rookie_prior_rows
)
   
df = df.merge(
    rookie_targets_priors,
    on=['season', 'position'],
    how='left'
)

# Sorting dataset into players and seasons (chronologically)
df = df.sort_values(['player_id','season','week']).reset_index(drop=True)

# Calculate average stat from up to the previous N (window) eligible games
def rolling_avg_with_last_season(df, stat, window):
    past_stats = []

    for games_back in range(1, window + 1):

        # Get player's stat from N games ago
        previous_value = (
            df.groupby('player_id')[stat]
            .shift(games_back)
        )

        # Get season that prev game occured in
        previous_season = (
            df.groupby('player_id')['season']
            .shift(games_back)
        )

        # Only allow games from current season or immediately prev season
        valid_history = (
            (previous_season == df['season']) |
            (previous_season == df['season'] - 1)
        )

        # Keep stat if game is eligible; otherwise replace with NaN
        past_stats.append(
            previous_value.where(valid_history)
        )

    # Put prev game stats side-by-side and get average across them
    return pd.concat(
        past_stats,
        axis=1
    ).mean(axis=1)

        player_display_name position  season  rookie_season  \
33421           Graham Gano        K    2020         2009.0   
33433        Rob Gronkowski       TE    2020         2010.0   
33452            A.J. Green       WR    2020         2011.0   
33582           Jordan Reed       TE    2020         2013.0   
33659       Jerick McKinnon       RB    2020         2014.0   
33880          Trent Taylor       WR    2020         2017.0   
33907           Marcus Kemp       WR    2020         2017.0   
33925             Jake Butt       TE    2020         2017.0   
33958           Josh Malone       WR    2020         2017.0   
34011      Jeremy McNichols       RB    2020         2017.0   
34033           Nick Keizer       TE    2020         2018.0   
34113        Brandon Powell       WR    2020         2018.0   
34316       Chris Streveler       QB    2020         2020.0   
34318        James Robinson       RB    2020         2020.0   
34319   Rodrigo Blankenship        K    2020         20

In [23]:
# ----------- WR/TE Feature Engineering -----------

receiving_stats = [
    'targets',
    'receptions',
    'receiving_yards',
    'target_share',
    'receiving_air_yards',
    'air_yards_share',
    'receiving_tds',
    'receiving_yards_after_catch'
]

# Create 3 game and 5 game averages
for stat in receiving_stats:
    df[f'{stat}_avg_3'] = rolling_avg_with_last_season(
        df,
        stat,
        3
    )

    df[f'{stat}_avg_5'] = rolling_avg_with_last_season(
        df,
        stat,
        5
    )

# Trends
df['targets_trend'] = df['targets_avg_3'] - df['targets_avg_5']
df['target_share_trend'] = df['target_share_avg_3'] - df['target_share_avg_5']
df['rec_yards_trend'] = df['receiving_yards_avg_3'] - df['receiving_yards_avg_5']
df['rec_air_yards_trend'] = df['receiving_air_yards_avg_3'] - df['receiving_air_yards_avg_5']

print(
    df[
        (df['position'] == 'WR') &
        (df['season'] == 2020) &
        (df['week'] == 1)
    ][
        [
            'player_display_name',
            'targets_avg_3',
            'targets_avg_5',
            'receiving_yards_avg_3',
            'receiving_yards_avg_5'
        ]
    ].head(10)
)


     player_display_name  targets_avg_3  targets_avg_5  receiving_yards_avg_3  \
256     Larry Fitzgerald       6.333333            6.6              45.000000   
1036            Ted Ginn       1.333333            2.2               7.666667   
1389      Danny Amendola       6.666667            7.2              46.333333   
1540      DeSean Jackson       5.000000            5.0              79.500000   
2473      Julian Edelman       6.000000            8.2              35.666667   
2804    Emmanuel Sanders       4.666667            5.8              31.666667   
2906       Andre Roberts       0.666667            0.4               2.333333   
3881          A.J. Green            NaN            NaN                    NaN   
4013         Julio Jones      16.000000           13.2             126.000000   
4755        Randall Cobb       5.000000            4.6              50.333333   

      receiving_yards_avg_5  
256                    42.2  
1036                   14.6  
1389              

In [4]:
# ----------- RB Feature Engineering -----------

df['opportunities'] = df['carries'] + df['targets']

# Past 3 game averages
df['carries_avg_3'] = df.groupby(['player_id','season'])['carries'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rushing_yards_avg_3'] = df.groupby(['player_id','season'])['rushing_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rushing_tds_avg_3'] = df.groupby(['player_id','season'])['rushing_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['opportunities_avg_3'] = df.groupby(['player_id','season'])['opportunities'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())


# Past 5 game averages
df['carries_avg_5'] = df.groupby(['player_id','season'])['carries'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rushing_yards_avg_5'] = df.groupby(['player_id','season'])['rushing_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rushing_tds_avg_5'] = df.groupby(['player_id','season'])['rushing_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['opportunities_avg_5'] = df.groupby(['player_id','season'])['opportunities'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['carries_trend'] = df['carries_avg_3'] - df['carries_avg_5']
df['rushing_yards_trend'] = df['rushing_yards_avg_3'] - df['rushing_yards_avg_5']
df['opportunities_trend'] = df['opportunities_avg_3'] - df['opportunities_avg_5']


In [5]:
# ----------- QB Feature Engineering -----------

# Past 3 game averages
df['completions_avg_3'] = df.groupby(['player_id', 'season'])['completions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['attempts_avg_3'] = df.groupby(['player_id', 'season'])['attempts'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_yards_avg_3'] = df.groupby(['player_id', 'season'])['passing_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_tds_avg_3'] = df.groupby(['player_id', 'season'])['passing_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_int_avg_3'] = df.groupby(['player_id', 'season'])['passing_interceptions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_air_yards_avg_3'] = df.groupby(['player_id', 'season'])['passing_air_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_first_downs_avg_3'] = df.groupby(['player_id', 'season'])['passing_first_downs'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['completions_avg_5'] = df.groupby(['player_id', 'season'])['completions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['attempts_avg_5'] = df.groupby(['player_id', 'season'])['attempts'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_yards_avg_5'] = df.groupby(['player_id', 'season'])['passing_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_tds_avg_5'] = df.groupby(['player_id', 'season'])['passing_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_int_avg_5'] = df.groupby(['player_id', 'season'])['passing_interceptions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_air_yards_avg_5'] = df.groupby(['player_id', 'season'])['passing_air_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_first_downs_avg_5'] = df.groupby(['player_id', 'season'])['passing_first_downs'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['attempts_trend'] = df['attempts_avg_3'] - df['attempts_avg_5']
df['passing_yards_trend'] = df['passing_yards_avg_3'] - df['passing_yards_avg_5']
df['passing_air_yards_trend'] = df['passing_air_yards_avg_3'] - df['passing_air_yards_avg_5']


In [6]:
# ----------- K Feature Engineering -----------

# Past 3 game averages
df['fg_att_avg_3'] = df.groupby(['player_id', 'season'])['fg_att'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_made_avg_3'] = df.groupby(['player_id', 'season'])['fg_made'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_long_avg_3'] = df.groupby(['player_id', 'season'])['fg_long'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_made_50_59_avg_3'] = df.groupby(['player_id', 'season'])['fg_made_50_59'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['pat_att_avg_3'] = df.groupby(['player_id', 'season'])['pat_att'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['pat_made_avg_3'] = df.groupby(['player_id', 'season'])['pat_made'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['fg_att_avg_5'] = df.groupby(['player_id', 'season'])['fg_att'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_made_avg_5'] = df.groupby(['player_id', 'season'])['fg_made'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_long_avg_5'] = df.groupby(['player_id', 'season'])['fg_long'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_made_50_59_avg_5'] = df.groupby(['player_id', 'season'])['fg_made_50_59'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['pat_att_avg_5'] = df.groupby(['player_id', 'season'])['pat_att'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['pat_made_avg_5'] = df.groupby(['player_id', 'season'])['pat_made'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['fg_att_trend'] = df['fg_att_avg_3'] - df['fg_att_avg_5']
df['fg_made_trend'] = df['fg_made_avg_3'] - df['fg_made_avg_5']

# Kicker fantasy points
df['kicker_fantasy_points'] = (
    3 * df['fg_made'] +
    1 * df['pat_made']
)


In [11]:
# ----------- Opponent Matchup Feature Engineering -----------

fantasy_positions = ['QB', 'RB', 'WR', 'TE', 'K']

matchup_df = df[
    df['position'].isin(fantasy_positions)
].copy()

matchup_df['matchup_points'] = matchup_df['fantasy_points_ppr']

matchup_df.loc[
    matchup_df['position'] == 'K',
    'matchup_points'
] = matchup_df.loc[
    matchup_df['position'] == 'K',
    'kicker_fantasy_points'
]

weekly_points_allowed = (
    matchup_df.groupby(
        ['season', 'week', 'opponent_team', 'position'],
        as_index=False
    )['matchup_points']
    .sum()
)

weekly_points_allowed = weekly_points_allowed.rename(
    columns = {
        'matchup_points': 'points_allowed'
    }
)

weekly_points_allowed = weekly_points_allowed.sort_values(
    ['opponent_team', 'position', 'season', 'week']
).reset_index(drop=True)

matchup_group = [
    'opponent_team',
    'position',
    'season'
]

weekly_points_allowed['opp_points_allowed_avg_3'] = (
    weekly_points_allowed
    .groupby(matchup_group)['points_allowed']
    .transform(
        lambda x:
        x.shift(1)
        .rolling(window=3, min_periods=1)
        .mean()
    )
)

weekly_points_allowed['opp_points_allowed_avg_5'] = (
    weekly_points_allowed
    .groupby(matchup_group)['points_allowed']
    .transform(
        lambda x:
        x.shift(1)
        .rolling(window=5, min_periods=1)
        .mean()
    )
)

weekly_points_allowed['opp_points_allowed_trend'] = (
    weekly_points_allowed['opp_points_allowed_avg_3']
    - weekly_points_allowed['opp_points_allowed_avg_5']
)

matchup_features = weekly_points_allowed[
    [
        'season',
        'week',
        'opponent_team',
        'position',
        'opp_points_allowed_avg_3',
        'opp_points_allowed_avg_5',
        'opp_points_allowed_trend'
    ]
]

df = df.merge(
    matchup_features,
    on=[
        'season',
        'week',
        'opponent_team',
        'position'
    ],
    how='left'
)

print(
    df[
        (df['position'] == 'WR') &
        (df['week'] > 1)
    ][
        [
            'player_display_name',
            'season',
            'week',
            'opponent_team',
            'opp_points_allowed_avg_3',
            'opp_points_allowed_avg_5',
            'opp_points_allowed_trend'
        ]
    ].head(15)
)

     player_display_name  season  week opponent_team  \
1038    Larry Fitzgerald    2019     2           BAL   
1070      Danny Amendola    2019     2           LAC   
1077      Matthew Slater    2019     2           MIA   
1085    Michael Crabtree    2019     2           BAL   
1095      Julian Edelman    2019     2           MIA   
1102    Emmanuel Sanders    2019     2           CHI   
1108       Antonio Brown    2019     2           MIA   
1116    Demaryius Thomas    2019     2           CLE   
1124         Julio Jones    2019     2           PHI   
1135        Randall Cobb    2019     2           WAS   
1148       Dwayne Harris    2019     2            KC   
1154         Chris Hogan    2019     2            TB   
1158     Dontrelle Inman    2019     2           DET   
1171        Cole Beasley    2019     2           NYG   
1183     Travis Benjamin    2019     2           DET   

      opp_points_allowed_avg_3  opp_points_allowed_avg_5  \
1038                      29.2             

In [13]:
# To Parquet
df.to_parquet(
    "../data/processed/player_features.parquet",
    index=False
)